# Model 3 — XGBoost Improved
**Member 1 · Probability targets: `y_24h`, `y_72h`**

Builds on `notebooks_model2/train_xgb_engineered.ipynb` (test ROC-AUC: 0.824 / 0.803).

### Diagnosis from Model 2
| Issue | Evidence | Fix |
|---|---|---|
| Moderate overfitting | train-val gap ~0.09 | Stronger regularisation + Optuna hyperparameter search |
| Unused physical signals | `gcmt_moment_exponent` ranked #4 but not log-transformed; eigenvalue plunge angles in top 20 | Better GCMT feature engineering |
| Weak seismicity-rate features | Omori law: aftershock rate ∝ prior activity | Omori-inspired interaction features |
| Suboptimal hyperparameters | Default `max_depth=5`, `lr=0.05` not tuned | Optuna search on val ROC-AUC |

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

# ---- robust repo root detection ----
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find repo root containing /src")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("XGBoost      =", xgb.__version__)
print("Optuna       =", optuna.__version__)

from src.evaluation import evaluate_binary_probabilities
from src.models.feature_sets import (
    BASE_TABULAR_FEATURES,
    GCMT_FEATURES,
    QUALITY_FEATURES,
)
from src.models.input_layer import (
    InputConfig,
    load_modeling_splits,
    prepare_tabular_inputs,
)
from src.utils.paths import METRICS_DIR

PROJECT_ROOT = /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10
XGBoost      = 3.2.0
Optuna       = 4.8.0


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = "earthquake_aftershock_v2_gcmt"
MODEL_NAME = "xgb_improved"
TARGETS = ["y_24h", "y_72h"]
OPTUNA_TRIALS = 60   # increase if you want a longer search

In [3]:
splits = load_modeling_splits(dataset_name=DATASET_NAME)

for split_name, df in splits.items():
    print(
        f"{split_name:5s}  {len(df):>5} rows  |  "
        f"y_24h pos: {df['y_24h'].mean():.3f}  "
        f"y_72h pos: {df['y_72h'].mean():.3f}"
    )

train  14968 rows  |  y_24h pos: 0.438  y_72h pos: 0.501
val     1461 rows  |  y_24h pos: 0.390  y_72h pos: 0.450
test    2052 rows  |  y_24h pos: 0.524  y_72h pos: 0.587


# Improved Feature Engineering

## 2 · Improved feature engineering

Three targeted additions on top of Model 2's feature set:

**Fix 1 — Better GCMT moment tensor features**
`gcmt_moment_exponent` was the #4 most important feature in Model 2 but was fed raw.
The physical quantity is `M₀ = mantissa × 10^exponent` — we now log-transform the
full scalar moment more carefully and add the exponent directly as a clean ordinal.

**Fix 2 — Eigenvalue geometry features**
`gcmt_eig2_plunge` (intermediate eigenvalue plunge) appeared in the top 20 for both
targets, but we only had `sin_dip` before. We now add all three principal axis plunge
angles and their interactions, which encode the 3-D fault geometry more completely.

**Fix 3 — Omori-inspired seismicity rate features**
The Omori–Utsu law says aftershock rate decays as ~1/t from prior activity.
`prior_global_event_count_24h` is already available; we add the 24h/7d ratio as a
seismicity-acceleration proxy and a magnitude-weighted activity interaction.

In [4]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Extended feature engineering. Returns a copy; does not modify input."""
    df = df.copy()
    has_gcmt = df["has_gcmt"].fillna(False)

    # ══ MODEL 2 FEATURES (kept unchanged) ════════════════════════════════════

    # Depth regime
    df["depth_shallow"] = (df["trigger_depth_km"] < 70).astype(float)
    df["depth_intermediate"] = (df["trigger_depth_km"].between(70, 300)).astype(float)
    df["depth_deep"] = (df["trigger_depth_km"] > 300).astype(float)

    # Seismic size
    df["log_magnitude"] = np.log10(df["trigger_magnitude"].clip(lower=1e-6))
    df["mag_depth_ratio"] = df["trigger_magnitude"] / (df["trigger_depth_km"] + 1)
    df["log_scalar_moment"] = np.where(
        has_gcmt & df["gcmt_scalar_moment"].notna(),
        np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)),
        np.nan,
    )

    # Temporal cyclical
    df["sin_month"] = np.sin(2 * np.pi * df["trigger_month"] / 12)
    df["cos_month"] = np.cos(2 * np.pi * df["trigger_month"] / 12)
    df["sin_hour"] = np.sin(2 * np.pi * df["trigger_hour"] / 24)
    df["cos_hour"] = np.cos(2 * np.pi * df["trigger_hour"] / 24)
    df["sin_dayofyear"] = np.sin(2 * np.pi * df["trigger_dayofyear"] / 365)
    df["cos_dayofyear"] = np.cos(2 * np.pi * df["trigger_dayofyear"] / 365)

    # Spatial
    df["ring_of_fire"] = (
        (np.abs(df["trigger_longitude"]) > 130)
        & (df["trigger_latitude"].between(-60, 60))
    ).astype(float)

    # Tectonic regime from rake
    def _rake_regimes(rake: float) -> tuple[float, float, float]:
        if pd.isna(rake):
            return np.nan, np.nan, np.nan
        r = rake % 360
        ss_dist = min(abs(r), abs(r - 180), abs(r - 360))
        return float(ss_dist < 45), float(45 <= r <= 135), float(225 <= r <= 315)

    regimes = df["rake"].apply(
        lambda r: _rake_regimes(r) if pd.notna(r) else (np.nan, np.nan, np.nan)
    )
    df["is_strike_slip"] = regimes.apply(lambda x: x[0]).where(has_gcmt)
    df["is_reverse"] = regimes.apply(lambda x: x[1]).where(has_gcmt)
    df["is_normal"] = regimes.apply(lambda x: x[2]).where(has_gcmt)

    df["sin_dip"] = np.sin(np.radians(df["dip"].where(has_gcmt)))
    e1 = df["gcmt_eig1"].where(has_gcmt)
    e2 = df["gcmt_eig2"].where(has_gcmt)
    e3 = df["gcmt_eig3"].where(has_gcmt)
    denom = (e1.abs() + e3.abs()).replace(0, np.nan)
    df["clvd_fraction"] = (2 * e2.abs() / denom).where(has_gcmt)
    df["centroid_depth_diff"] = (df["gcmt_depth_km"] - df["trigger_depth_km"]).where(has_gcmt)
    df["log_half_duration"] = np.log1p(df["gcmt_half_duration_sec"].where(has_gcmt))
    df["mag_diff_abs"] = df["gcmt_mag_diff"].abs().where(has_gcmt)

    # ══ FIX 1 — Better moment tensor magnitude features ══════════════════════
    exp = df["gcmt_moment_exponent"].where(has_gcmt)
    df["moment_exponent_centered"] = (exp - 24.0).where(has_gcmt)

    df["gcmt_mw"] = np.where(
        has_gcmt & df["gcmt_scalar_moment"].notna(),
        (2 / 3) * np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)) - 10.7,
        np.nan,
    )
    df["mw_trigger_diff"] = (df["gcmt_mw"] - df["trigger_magnitude"]).where(has_gcmt)

    # ══ FIX 2 — Full eigenvalue geometry ═════════════════════════════════════
    p1 = df["gcmt_eig1_plunge"].where(has_gcmt)
    p3 = df["gcmt_eig3_plunge"].where(has_gcmt)

    df["sin_eig1_plunge"] = np.sin(np.radians(p1))
    df["cos_eig1_plunge"] = np.cos(np.radians(p1))
    df["sin_eig3_plunge"] = np.sin(np.radians(p3))
    df["cos_eig3_plunge"] = np.cos(np.radians(p3))

    df["tp_plunge_diff"] = (p1 - p3).where(has_gcmt)
    df["eig_ratio"] = (e1.abs() / (e1.abs() + e3.abs()).replace(0, np.nan)).where(has_gcmt)

    # ══ FIX 3 — Omori-inspired seismicity rate features ══════════════════════
    prior_24h = df["prior_global_event_count_24h"]
    prior_7d = df["prior_global_event_count_7d"]

    df["seismicity_acceleration"] = prior_24h / (prior_7d / 7.0).replace(0, np.nan)
    df["log_prior_24h"] = np.log1p(prior_24h)
    df["log_prior_7d"] = np.log1p(prior_7d)
    df["mag_x_log_prior"] = df["trigger_magnitude"] * np.log1p(prior_24h)
    df["mag_x_shallow"] = df["trigger_magnitude"] * df["depth_shallow"]

    return df


splits_eng = {name: engineer_features(df) for name, df in splits.items()}
print("Feature engineering complete.")
print("Columns after engineering:", splits_eng["train"].shape[1])

Feature engineering complete.
Columns after engineering: 127


# Determine Feature Set

In [5]:
MODEL2_FEATURES = [
    "depth_shallow", "depth_intermediate", "depth_deep",
    "log_magnitude", "mag_depth_ratio", "log_scalar_moment",
    "sin_month", "cos_month", "sin_hour", "cos_hour",
    "sin_dayofyear", "cos_dayofyear", "ring_of_fire",
    "is_strike_slip", "is_reverse", "is_normal",
    "sin_dip", "clvd_fraction", "centroid_depth_diff",
    "log_half_duration", "mag_diff_abs",
]

MODEL3_NEW_FEATURES = [
    "moment_exponent_centered", "gcmt_mw", "mw_trigger_diff",
    "sin_eig1_plunge", "cos_eig1_plunge",
    "sin_eig3_plunge", "cos_eig3_plunge",
    "tp_plunge_diff", "eig_ratio",
    "seismicity_acceleration", "log_prior_24h", "log_prior_7d",
    "mag_x_log_prior", "mag_x_shallow",
]

_seen = set()
FULL_FEATURE_SET = []
for f in (BASE_TABULAR_FEATURES + QUALITY_FEATURES + GCMT_FEATURES + MODEL2_FEATURES + MODEL3_NEW_FEATURES):
    if f not in _seen:
        FULL_FEATURE_SET.append(f)
        _seen.add(f)

print("Total requested features:", len(FULL_FEATURE_SET))
print("New features added vs Model 2:", len(MODEL3_NEW_FEATURES))

Total requested features: 91
New features added vs Model 2: 14


# Prepare Shared Pipeline

In [6]:
prepared = {}

for target in TARGETS:
    config = InputConfig(
        feature_cols=FULL_FEATURE_SET,
        target_col=target,
        missing_strategy="none",   # XGBoost handles NaN natively
        scale=False,
        allow_missing_optional=True,
        drop_rows_with_missing_target=True,
    )
    inputs = prepare_tabular_inputs(config=config, splits=splits_eng)
    prepared[target] = inputs
    print(
        f"{target} — X_train: {inputs.X_train.shape}, "
        f"resolved features: {len(inputs.feature_cols)}"
    )

y_24h — X_train: (14968, 91), resolved features: 91
y_72h — X_train: (14968, 91), resolved features: 91


# Optuna Objective

In [7]:
def make_objective(inp):
    def objective(trial: optuna.Trial) -> float:
        params = dict(
            n_estimators=600,
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            max_depth=trial.suggest_int("max_depth", 3, 7),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_weight=trial.suggest_int("min_child_weight", 5, 50),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            tree_method="hist",
            missing=np.nan,
            eval_metric="logloss",
            early_stopping_rounds=30,
            random_state=42,
            n_jobs=-1,
        )

        model = xgb.XGBClassifier(**params)
        model.fit(
            inp.X_train,
            inp.y_train,
            eval_set=[(inp.X_val, inp.y_val)],
            verbose=False,
        )
        val_prob = model.predict_proba(inp.X_val)[:, 1]
        return roc_auc_score(inp.y_val, val_prob)

    return objective

# Optuna Run

In [8]:
best_params = {}

for target in TARGETS:
    print(f"Tuning {target} ({OPTUNA_TRIALS} trials) ...")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(
        make_objective(prepared[target]),
        n_trials=OPTUNA_TRIALS,
        show_progress_bar=False,
    )

    best_params[target] = study.best_params
    print(f"  best val ROC-AUC : {study.best_value:.4f}")
    print(f"  best params      : {study.best_params}")
    print()

Tuning y_24h (60 trials) ...
  best val ROC-AUC : 0.8056
  best params      : {'learning_rate': 0.06600565479842768, 'max_depth': 5, 'subsample': 0.9037070133190904, 'colsample_bytree': 0.7556987268537234, 'min_child_weight': 13, 'reg_alpha': 0.03134769562453638, 'reg_lambda': 0.12279188124366536}

Tuning y_72h (60 trials) ...
  best val ROC-AUC : 0.7766
  best params      : {'learning_rate': 0.0838001258274829, 'max_depth': 6, 'subsample': 0.9462878828588784, 'colsample_bytree': 0.8791118085065368, 'min_child_weight': 31, 'reg_alpha': 7.534933229670127, 'reg_lambda': 0.0011065136021090993}



# Train Final Models

In [9]:
models = {}

for target in TARGETS:
    inp = prepared[target]
    params = dict(
        **best_params[target],
        n_estimators=1000,
        tree_method="hist",
        missing=np.nan,
        eval_metric="logloss",
        early_stopping_rounds=40,
        random_state=42,
        n_jobs=-1,
    )

    model = xgb.XGBClassifier(**params)
    model.fit(
        inp.X_train,
        inp.y_train,
        eval_set=[(inp.X_val, inp.y_val)],
        verbose=False,
    )
    models[target] = model

    val_auc = roc_auc_score(inp.y_val, model.predict_proba(inp.X_val)[:, 1])
    test_auc = roc_auc_score(inp.y_test, model.predict_proba(inp.X_test)[:, 1])
    print(
        f"{target} — best_iter: {model.best_iteration:4d}  |  "
        f"val ROC-AUC: {val_auc:.4f}  |  test ROC-AUC: {test_auc:.4f}"
    )

y_24h — best_iter:  220  |  val ROC-AUC: 0.8056  |  test ROC-AUC: 0.8200
y_72h — best_iter:  256  |  val ROC-AUC: 0.7766  |  test ROC-AUC: 0.7889


# Evaluate on all splits

In [10]:
summary_rows = []

for target in TARGETS:
    inp = prepared[target]
    model = models[target]
    horizon = int(target.split("_")[1].replace("h", ""))

    for split_name, X, y in [
        ("train", inp.X_train, inp.y_train),
        ("val", inp.X_val, inp.y_val),
        ("test", inp.X_test, inp.y_test),
    ]:
        y_prob = model.predict_proba(X)[:, 1]
        metrics = evaluate_binary_probabilities(y_true=y, y_prob=y_prob)
        summary_rows.append({
            "model_name": MODEL_NAME,
            "split": split_name,
            "horizon": horizon,
            **metrics,
        })

_split_order = {"train": 0, "val": 1, "test": 2}
metrics_df = (
    pd.DataFrame(summary_rows)
    .assign(_o=lambda d: d["split"].map(_split_order))
    .sort_values(["model_name", "horizon", "_o"])
    .drop(columns="_o")
    .reset_index(drop=True)
)

metrics_df

,model_name,split,horizon,brier_score,log_loss,roc_auc,n_obs,positive_rate
0,xgb_improved,train,24,0.128178,0.408288,0.906477,14968,0.438268
1,xgb_improved,val,24,0.169742,0.513679,0.805622,1461,0.390144
2,xgb_improved,test,24,0.176957,0.528116,0.819980,2052,0.524366
3,xgb_improved,train,72,0.122172,0.395865,0.922043,14968,0.501203
4,xgb_improved,val,72,0.188406,0.556820,0.776624,1461,0.450376
5,xgb_improved,test,72,0.188223,0.557163,0.788880,2052,0.587232


# Compare on previous models

In [11]:
model2_path = METRICS_DIR / "xgb_engineered_metrics.csv"
logreg_path = METRICS_DIR / "logreg_baseline_metrics.csv"
team_base_path = METRICS_DIR / "baselines_metrics.csv"

frames = [metrics_df]

if model2_path.exists():
    m2 = pd.read_csv(model2_path)
    if "target" in m2.columns and "horizon" not in m2.columns:
        m2["horizon"] = m2["target"].str.extract(r"(\d+)").astype(int)
    frames.append(m2)

if logreg_path.exists():
    lr = pd.read_csv(logreg_path)
    if "target" in lr.columns and "horizon" not in lr.columns:
        lr["horizon"] = lr["target"].str.extract(r"(\d+)").astype(int)
    frames.append(lr)

if team_base_path.exists():
    tb = pd.read_csv(team_base_path)
    frames.append(tb)

comparison = pd.concat(frames, ignore_index=True)

pivot = (
    comparison
    .pivot_table(index=["horizon", "split"], columns="model_name", values="roc_auc")
    .round(4)
)

if "xgb_engineered" in pivot.columns and MODEL_NAME in pivot.columns:
    pivot["Δ vs model2"] = (pivot[MODEL_NAME] - pivot["xgb_engineered"]).round(4)

print("ROC-AUC comparison:")
print(pivot.to_string())

ROC-AUC comparison:
model_name     climatology  simplified_rj  xgb_improved
horizon split                                          
24      test           0.5         0.5556        0.8200
        train          0.5         0.5869        0.9065
        val            0.5         0.5828        0.8056
72      test           0.5         0.5521        0.7889
        train          0.5         0.5836        0.9220
        val            0.5         0.5807        0.7766


# Feature Importance

In [12]:
for target in TARGETS:
    model = models[target]
    features = prepared[target].feature_cols

    imp = (
        pd.DataFrame({"feature": features, "importance": model.feature_importances_})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    imp["new"] = imp["feature"].isin(MODEL3_NEW_FEATURES).map({True: "★", False: ""})

    print(f"\n=== {target} — top 20 features (★ = new in Model 3) ===")
    print(imp.head(20).to_string(index=False))


=== y_24h — top 20 features (★ = new in Model 3) ===
                     feature  importance new
               gcmt_depth_km    0.046589    
               mag_x_shallow    0.046182   ★
             mag_x_log_prior    0.044002   ★
              is_strike_slip    0.043032    
             mag_depth_ratio    0.040790    
      gcmt_half_duration_sec    0.030610    
           log_half_duration    0.028317    
        gcmt_moment_exponent    0.026857    
          depth_intermediate    0.023948    
    moment_exponent_centered    0.023122   ★
            gcmt_eig2_plunge    0.019155    
           trigger_magnitude    0.019069    
              gcmt_magnitude    0.018529    
prior_global_event_count_24h    0.017103    
         gcmt_depth_error_km    0.016299    
             sin_eig1_plunge    0.015625   ★
            trigger_depth_km    0.015194    
                        dmin    0.015150    
                        dip2    0.014069    
                         dip    0.012924    



# Build prediction table

In [13]:
prediction_rows = []

for target in TARGETS:
    inp = prepared[target]
    model = models[target]
    horizon = int(target.split("_")[1].replace("h", ""))

    for split_name, X, y, raw_df in [
        ("train", inp.X_train, inp.y_train, splits_eng["train"]),
        ("val", inp.X_val, inp.y_val, splits_eng["val"]),
        ("test", inp.X_test, inp.y_test, splits_eng["test"]),
    ]:
        y_prob = model.predict_proba(X)[:, 1]
        prediction_rows.append(pd.DataFrame({
            "trigger_event_id": raw_df.loc[X.index, "trigger_event_id"].values,
            "split": split_name,
            "horizon": horizon,
            "model_name": MODEL_NAME,
            "y_true": y.values,
            "y_prob": y_prob,
        }))

predictions_df = pd.concat(prediction_rows, ignore_index=True)

METRICS_DIR.mkdir(parents=True, exist_ok=True)
pred_path = METRICS_DIR / f"{MODEL_NAME}_predictions.csv"
metrics_path = METRICS_DIR / f"{MODEL_NAME}_metrics.csv"

predictions_df.to_csv(pred_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

print("Predictions →", pred_path)
print("Metrics     →", metrics_path)
print("Rows saved  :", len(predictions_df))

Predictions → /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/xgb_improved_predictions.csv
Metrics     → /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/xgb_improved_metrics.csv
Rows saved  : 36962
